In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [ ]:
df_train = pd.read_csv('../data/train.csv')
df_train.head()

In [ ]:
df_test = pd.read_csv('../data/test.csv')
df_test.head()

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
df_train.describe()

In [ ]:
df_test.describe()

In [ ]:
print(df_train['class'].unique())

In [ ]:
# Class balance — balanced_accuracy is the metric, so imbalance matters
print("Class counts:")
print(df_train['class'].value_counts())
print("\nClass proportions:")
print(df_train['class'].value_counts(normalize=True).round(4))

In [ ]:
# Inspect the two categorical columns
for col in ['spectral_type', 'galaxy_population']:
    print(f"=== {col} ===")
    print("train unique:", sorted(df_train[col].unique().tolist()))
    print("test  unique:", sorted(df_test[col].unique().tolist()))
    print("n train cats:", df_train[col].nunique(), "| n test cats:", df_test[col].nunique())
    print()

# Deep EDA — hunt inconsistencies before modeling

In [ ]:
# --- Block 1: impossible/negative values + duplicates ---
mag_cols = ['u', 'g', 'r', 'i', 'z']

print("### Negative magnitudes (physically suspicious) ###")
for c in mag_cols:
    n_tr = (df_train[c] < 0).sum()
    n_te = (df_test[c] < 0).sum()
    print(f"{c}: train={n_tr}  test={n_te}")

print("\n### Negative redshift (z < 0; tiny negatives = measurement noise) ###")
print("train:", (df_train['redshift'] < 0).sum(), " test:", (df_test['redshift'] < 0).sum())
print("min redshift train:", df_train['redshift'].min(), " test:", df_test['redshift'].min())

print("\n### Extreme magnitude outliers (mag > 30 or < 5 = likely junk) ###")
for c in mag_cols:
    bad = ((df_train[c] > 30) | (df_train[c] < 5)).sum()
    print(f"{c}: {bad}")

print("\n### Duplicate rows (ignoring id) ###")
feat_cols = [c for c in df_train.columns if c not in ('id', 'class')]
print("train full-feature dups:", df_train.duplicated(subset=feat_cols).sum())
print("test  full-feature dups:", df_test.duplicated(subset=feat_cols).sum())

In [ ]:
# Look at the suspicious u rows before deciding to clip
# (negative u OR u > 30 OR u < 5)
mask = (df_train['u'] < 0) | (df_train['u'] > 30) | (df_train['u'] < 5)
print("n suspicious u rows:", mask.sum())
display(df_train.loc[mask])

# also: how do these u values compare to the same row's other bands?
# physically u should be roughly in line with g,r,i,z (within a few mag)
print("\nFor context, normal u range (1st/99th pct):",
      df_train['u'].quantile(0.01).round(3), "to", df_train['u'].quantile(0.99).round(3))

In [ ]:
# --- Winsorize magnitudes to [0.1, 99.9] percentile ---
# Bounds computed on TRAIN ONLY (they are parameters -> using test would leak).
# Applied identically to train and test. redshift left untouched (high-z is real signal).

mag_cols = ['u', 'g', 'r', 'i', 'z']

clip_bounds = {}
for c in mag_cols:
    lo = df_train[c].quantile(0.001)
    hi = df_train[c].quantile(0.999)
    clip_bounds[c] = (lo, hi)

print("Clip bounds (from train):")
for c, (lo, hi) in clip_bounds.items():
    print(f"  {c}: [{lo:.3f}, {hi:.3f}]")

# how many values get clipped, per split
print("\nValues clipped:")
for c, (lo, hi) in clip_bounds.items():
    tr = ((df_train[c] < lo) | (df_train[c] > hi)).sum()
    te = ((df_test[c]  < lo) | (df_test[c]  > hi)).sum()
    print(f"  {c}: train={tr}  test={te}")

# apply
for c, (lo, hi) in clip_bounds.items():
    df_train[c] = df_train[c].clip(lo, hi)
    df_test[c]  = df_test[c].clip(lo, hi)

# verify the 3 broken-u rows are fixed
print("\nAfter clip, former suspicious-u rows:")
display(df_train.loc[[78641, 251744, 523685], ['u','g','r','i','z','redshift','class']])
print("\nu range now:", df_train['u'].min().round(3), "to", df_train['u'].max().round(3))

In [ ]:
# --- Block 2: redshift by class (the strongest single feature) ---
print("Redshift stats per class:")
display(df_train.groupby('class')['redshift'].describe().round(4))

# overlap check: how often does redshift alone separate classes?
print("\nredshift quantiles per class (5%, 25%, 50%, 75%, 95%):")
for cls in ['STAR', 'GALAXY', 'QSO']:
    q = df_train.loc[df_train['class']==cls, 'redshift'].quantile([.05,.25,.5,.75,.95]).round(4)
    print(f"  {cls:7s}: {q.values}")

fig, ax = plt.subplots(1, 2, figsize=(14,4))
for cls in ['STAR', 'GALAXY', 'QSO']:
    sub = df_train.loc[df_train['class']==cls, 'redshift']
    ax[0].hist(sub, bins=100, alpha=0.5, label=cls, density=True)
    ax[1].hist(sub.clip(-0.05, 1.5), bins=100, alpha=0.5, label=cls, density=True)
ax[0].set_title('redshift (full range)'); ax[0].legend(); ax[0].set_xlabel('redshift')
ax[1].set_title('redshift (zoomed -0.05 to 1.5)'); ax[1].legend(); ax[1].set_xlabel('redshift')
plt.tight_layout(); plt.show()

In [ ]:
# --- Block 3: colors (adjacent-band differences) by class ---
# These are EDA-only here (computed on a copy). We add them for-real in the
# feature step later. Colors capture spectrum shape -> resolve redshift overlap.

eda = df_train.copy()
eda['u_g'] = eda['u'] - eda['g']
eda['g_r'] = eda['g'] - eda['r']
eda['r_i'] = eda['r'] - eda['i']
eda['i_z'] = eda['i'] - eda['z']
color_cols = ['u_g', 'g_r', 'r_i', 'i_z']

print("Mean color per class:")
display(eda.groupby('class')[color_cols].mean().round(3))
print("\nStd color per class:")
display(eda.groupby('class')[color_cols].std().round(3))

fig, ax = plt.subplots(2, 2, figsize=(14,8))
for axi, col in zip(ax.ravel(), color_cols):
    for cls in ['STAR', 'GALAXY', 'QSO']:
        sub = eda.loc[eda['class']==cls, col]
        axi.hist(sub.clip(sub.quantile(0.01), sub.quantile(0.99)),
                 bins=80, alpha=0.5, label=cls, density=True)
    axi.set_title(col); axi.legend(); axi.set_xlabel(col)
plt.tight_layout(); plt.show()

# Feature build + encoding

Assemble the final matrix from what EDA showed mattered:
- **raw mags** `u,g,r,i,z` + **redshift** (strongest)
- **colors** `u_g, g_r, r_i, i_z` (resolve overlap; g_r/u_g/r_i strongest)
- **one-hot** `spectral_type` (4) + `galaxy_population` (2)
- drop `id`, keep `alpha`/`delta` for now (test their value later)

Target `class` → integer codes. Same transform applied to train and test.

In [ ]:
def build_features(df):
    out = df.copy()
    # colors (spectrum shape)
    out['u_g'] = out['u'] - out['g']
    out['g_r'] = out['g'] - out['r']
    out['r_i'] = out['r'] - out['i']
    out['i_z'] = out['i'] - out['z']
    # one-hot categoricals (drop_first=False -> keep all, linear models + scaling handle it)
    out = pd.get_dummies(out, columns=['spectral_type', 'galaxy_population'], dtype=float)
    return out

train_fe = build_features(df_train)
test_fe  = build_features(df_test)

# target -> integer codes (sorted: GALAXY=0, QSO=1, STAR=2)
classes = sorted(df_train['class'].unique())          # ['GALAXY','QSO','STAR']
class_to_int = {c: i for i, c in enumerate(classes)}
int_to_class = {i: c for c, i in class_to_int.items()}
print("class mapping:", class_to_int)

y = df_train['class'].map(class_to_int).values

# feature columns = everything except id and target
drop_cols = ['id', 'class']
feature_cols = [c for c in train_fe.columns if c not in drop_cols]

# align test to same columns (in case a dummy is missing in one split)
X      = train_fe[feature_cols].copy()
X_test = test_fe.reindex(columns=feature_cols, fill_value=0.0).copy()

print("\nn features:", len(feature_cols))
print("feature_cols:", feature_cols)
print("\nX shape:", X.shape, "| X_test shape:", X_test.shape, "| y shape:", y.shape)
print("any NaN in X?", X.isna().any().any(), "| in X_test?", X_test.isna().any().any())
X.head()

# Baseline: scaled Logistic Regression

- Scaling **inside** a `Pipeline` so the scaler is fit per-fold on training data only — no leakage into validation.
- `StratifiedKFold` keeps the 65/20/14 class ratio in every fold.
- `class_weight='balanced'` because the metric (balanced_accuracy) weights all three classes equally, so we must not let GALAXY dominate.
- Scored with **balanced_accuracy** = the competition metric.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score, make_scorer, classification_report, confusion_matrix
import time

bal_acc = make_scorer(balanced_accuracy_score)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        solver='lbfgs',
        n_jobs=-1,
        random_state=42,
    )),
])

t0 = time.time()
scores = cross_val_score(logreg_pipe, X, y, cv=skf, scoring=bal_acc, n_jobs=-1)
print(f"LogReg baseline — balanced_accuracy per fold: {np.round(scores, 4)}")
print(f"mean = {scores.mean():.4f}  |  std = {scores.std():.4f}")
print(f"(took {time.time()-t0:.1f}s)")

In [ ]:
# --- Per-class diagnostic: where does the 9% error live? ---
from sklearn.model_selection import train_test_split

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

logreg_pipe.fit(X_tr, y_tr)
pred_va = logreg_pipe.predict(X_va)

print("balanced_accuracy (holdout):", round(balanced_accuracy_score(y_va, pred_va), 4))
print("\nclassification_report (target_names by code 0/1/2):")
print(classification_report(y_va, pred_va, target_names=[int_to_class[i] for i in [0,1,2]]))

cm = confusion_matrix(y_va, pred_va)
cm_df = pd.DataFrame(cm,
                     index=[f"true_{int_to_class[i]}" for i in [0,1,2]],
                     columns=[f"pred_{int_to_class[i]}" for i in [0,1,2]])
print("Confusion matrix (rows=true, cols=pred):")
display(cm_df)

# per-class recall = the thing balanced_accuracy averages
recall_per_class = cm.diagonal() / cm.sum(axis=1)
for i in [0,1,2]:
    print(f"recall {int_to_class[i]:7s}: {recall_per_class[i]:.4f}")

# Tune LogReg: class_weight + C

GALAXY (recall 0.866) leaks into STAR/QSO because `class_weight='balanced'` over-boosts the small classes (STAR precision only 0.66). Sweep:
- **class_weight**: `None`, `'balanced'`, and custom dicts that boost STAR/QSO *less* aggressively.
- **C**: regularization strength (lower C = more reg, smoother boundary).

Scored on the same 5-fold stratified CV with balanced_accuracy.

In [ ]:
# class_weight options (codes: GALAXY=0, QSO=1, STAR=2)
cw_options = {
    'none':          None,
    'balanced':      'balanced',
    'mild_boost':    {0: 1.0, 1: 1.5, 2: 2.0},   # gently favor QSO/STAR
    'medium_boost':  {0: 1.0, 1: 2.0, 2: 3.0},
    'galaxy_favor':  {0: 1.3, 1: 1.0, 2: 1.0},   # protect GALAXY recall
}
C_options = [0.1, 0.5, 1.0, 3.0]

results = []
t0 = time.time()
for cw_name, cw in cw_options.items():
    for C in C_options:
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression(
                class_weight=cw, C=C, max_iter=2000,
                solver='lbfgs', random_state=42)),
        ])
        sc = cross_val_score(pipe, X, y, cv=skf, scoring=bal_acc, n_jobs=-1)
        results.append((cw_name, C, sc.mean(), sc.std()))
        print(f"cw={cw_name:13s} C={C:<4} -> bal_acc={sc.mean():.4f} (±{sc.std():.4f})")

print(f"\n(total {time.time()-t0:.1f}s)")
res_df = pd.DataFrame(results, columns=['class_weight','C','mean_bal_acc','std']).sort_values('mean_bal_acc', ascending=False)
print("\nTop 5:")
display(res_df.head())

# Stacking ensemble → TabPFN-3 meta-learner

LogReg caps ~0.913 because the class boundary is **nonlinear**. To reach the ~0.97 ceiling we stack:

```
Layer 0 (base models, 5-fold OOF):   XGBoost · CatBoost · PyTorch MLP
        ↓ out-of-fold class probabilities (no leakage)
Layer 1 (meta):                      TabPFN-3 over [base OOF logits + raw features]
        ↓
Final prediction → submission.csv
```

**Why OOF:** each training row is predicted only by models that did **not** see it (the other 4 folds), so the stack features are leak-free. Test probabilities = average of the 5 fold-models.

**Lessons baked in from the reference notebooks:**
- **No aggressive class prior** — a STAR×4.6 prior *lowered* score there. Use light/no reweighting; let `balance_probabilities` and the metric do the work.
- STAR is the weak-recall class to watch.

In [ ]:
# --- Stacking infrastructure ---
# X, X_test (DataFrames, 18 features), y (int codes) already exist from feature step.
# int_to_class / class_to_int already defined.

N_CLASSES = 3
N_FOLDS   = 5
SEED      = 42
oof_skf   = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# numpy views
X_np      = X.values.astype('float32')
X_test_np = X_test.values.astype('float32')
y_np      = y.astype('int64')
print("X_np:", X_np.shape, "| X_test_np:", X_test_np.shape, "| y_np:", y_np.shape)

# registries for base-model OOF / test probabilities
oof_probs  = {}   # name -> (N_train, 3) out-of-fold class probs
test_probs = {}   # name -> (N_test, 3) test class probs (avg over folds)

def run_oof(name, make_model, fit_predict, X_full=X_np, X_te=X_test_np, y_full=y_np):
    """Generic 5-fold OOF runner.
    make_model() -> fresh model
    fit_predict(model, Xtr, ytr, Xval) -> proba (n,3); also used on test.
    Returns oof (N,3), test_mean (M,3), prints per-fold + overall balanced_acc.
    """
    oof  = np.zeros((len(y_full), N_CLASSES), dtype='float32')
    test = np.zeros((X_te.shape[0], N_CLASSES), dtype='float32')
    t0 = time.time()
    for f, (tr, va) in enumerate(oof_skf.split(X_full, y_full)):
        model = make_model()
        oof[va] = fit_predict(model, X_full[tr], y_full[tr], X_full[va])
        test   += fit_predict(model, X_full[tr], y_full[tr], X_te) / N_FOLDS
        ba = balanced_accuracy_score(y_full[va], oof[va].argmax(1))
        print(f"  [{name}] fold {f+1}/{N_FOLDS}  bal_acc={ba:.4f}  ({time.time()-t0:.0f}s)")
    overall = balanced_accuracy_score(y_full, oof.argmax(1))
    print(f"  [{name}] OOF balanced_acc = {overall:.4f}   total {time.time()-t0:.0f}s")
    oof_probs[name]  = oof
    test_probs[name] = test
    return oof, test

In [ ]:
# --- Base model 1: XGBoost (GPU) ---
import xgboost as xgb

def make_xgb():
    return xgb.XGBClassifier(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_weight=3,
        reg_lambda=1.0,
        objective='multi:softprob',
        num_class=N_CLASSES,
        tree_method='hist',
        device='cuda',
        eval_metric='mlogloss',
        random_state=SEED,
        n_jobs=-1,
    )

def xgb_fit_predict(model, Xtr, ytr, Xpred):
    model.fit(Xtr, ytr, verbose=False)
    return model.predict_proba(Xpred)

print("Training XGBoost (5-fold OOF, GPU)...")
oof_xgb, test_xgb = run_oof('xgb', make_xgb, xgb_fit_predict)

In [ ]:
# --- Base model 2: CatBoost (GPU) ---
from catboost import CatBoostClassifier

def make_cat():
    return CatBoostClassifier(
        iterations=1200,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        loss_function='MultiClass',
        eval_metric='MultiClass',
        task_type='GPU',
        devices='0',
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
    )

def cat_fit_predict(model, Xtr, ytr, Xpred):
    model.fit(Xtr, ytr)
    return model.predict_proba(Xpred)

print("Training CatBoost (5-fold OOF, GPU)...")
oof_cat, test_cat = run_oof('cat', make_cat, cat_fit_predict)

In [ ]:
# --- Base model 3: PyTorch MLP (GPU) ---
# Neural net adds non-tree diversity. Needs SCALED input (scaler fit per-fold inside OOF).
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class MLP(nn.Module):
    def __init__(self, in_dim, n_cls=3, hidden=(256, 128, 64), p=0.2):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p)]
            d = h
        layers += [nn.Linear(d, n_cls)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def mlp_fit_predict(_unused, Xtr, ytr, Xpred):
    # scale inside the fold (fit on train part only)
    sc = StandardScaler().fit(Xtr)
    Xtr_s  = sc.transform(Xtr).astype('float32')
    Xpr_s  = sc.transform(Xpred).astype('float32')

    torch.manual_seed(SEED); np.random.seed(SEED)
    model = MLP(Xtr_s.shape[1]).to(torch_device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # mild class weights (NOT aggressive — refs warned heavy priors hurt)
    cls_w = torch.tensor(
        np.sqrt(len(ytr) / (N_CLASSES * np.bincount(ytr))), dtype=torch.float32
    ).to(torch_device)
    crit = nn.CrossEntropyLoss(weight=cls_w)

    ds = TensorDataset(torch.from_numpy(Xtr_s), torch.from_numpy(ytr))
    dl = DataLoader(ds, batch_size=4096, shuffle=True, drop_last=False)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3,
                                                steps_per_epoch=len(dl), epochs=25)
    model.train()
    for _ in range(25):
        for xb, yb in dl:
            xb, yb = xb.to(torch_device), yb.to(torch_device)
            opt.zero_grad()
            loss = crit(model(xb), yb)
            loss.backward(); opt.step(); sched.step()

    model.eval()
    with torch.no_grad():
        xt = torch.from_numpy(Xpr_s).to(torch_device)
        proba = torch.softmax(model(xt), dim=1).cpu().numpy()
    return proba

def make_mlp():
    return None  # model built inside fit_predict (needs scaled dims)

print(f"Training MLP (5-fold OOF, {torch_device})...")
oof_mlp, test_mlp = run_oof('mlp', make_mlp, mlp_fit_predict)

In [ ]:
# --- Base-model diversity + simple-average sanity blend ---
names = ['xgb', 'cat', 'mlp']
print("Base OOF balanced_acc:")
for n in names:
    print(f"  {n}: {balanced_accuracy_score(y_np, oof_probs[n].argmax(1)):.4f}")

# prediction agreement (share of identical argmax labels) between each pair
print("\nPairwise label agreement (lower = more diverse = better for stacking):")
preds = {n: oof_probs[n].argmax(1) for n in names}
for i in range(len(names)):
    for j in range(i+1, len(names)):
        agree = (preds[names[i]] == preds[names[j]]).mean()
        print(f"  {names[i]} vs {names[j]}: {agree:.4f}")

# simple average blend (free baseline before the TabPFN meta)
avg = sum(oof_probs[n] for n in names) / len(names)
print(f"\nSimple-average blend OOF balanced_acc: {balanced_accuracy_score(y_np, avg.argmax(1)):.4f}")

## TabPFN-3 meta-learner

Stack features = **base-model OOF logits** (XGB/Cat/MLP, 9 cols) **+ raw features** (18 cols) = 27 cols.
TabPFN-3 does in-context learning over these to find the best combination — especially on the
rows where the base models disagree.

Set `TABPFN_TOKEN` first (license `tabpfn-3-license-v1.0` must be accepted on ux.priorlabs.ai).
6 GB VRAM is tight for a 577 k-row context, so we validate on a holdout first; if it OOMs we
fall back to a smaller context / subsample.

In [ ]:
# --- Build meta features: base OOF logits + raw features ---
import os

# Load TABPFN_TOKEN from .env (gitignored) — never hardcode the key in the notebook.
if not os.environ.get("TABPFN_TOKEN"):
    from pathlib import Path
    env_path = Path('../.env')
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip())
assert os.environ.get("TABPFN_TOKEN"), \
    "TABPFN_TOKEN not set — put it in ../.env as TABPFN_TOKEN=... (file is gitignored)"
print("TABPFN_TOKEN loaded:", os.environ["TABPFN_TOKEN"][:12] + "...")

# EPS=1e-6 (not 1e-15): float32 can't represent 1 - 1e-15, which gives 1-p=0 -> div-by-zero.
EPS = 1e-6
def to_logit(p):
    p = np.clip(p, EPS, 1 - EPS)
    return np.log(p / (1 - p)).astype('float32')

# train meta = OOF logits (leak-free) + raw features
meta_train = np.concatenate(
    [to_logit(oof_probs[n]) for n in names] + [X_np], axis=1).astype('float32')
# test meta = averaged test logits + raw features
meta_test = np.concatenate(
    [to_logit(test_probs[n]) for n in names] + [X_test_np], axis=1).astype('float32')

print("meta_train:", meta_train.shape, "| meta_test:", meta_test.shape)
print("(9 base-logit cols + 18 raw = 27 features)")
print("any inf?", np.isinf(meta_train).any(), np.isinf(meta_test).any())

In [ ]:
# --- TabPFN-3 meta: validate on a holdout first (6 GB VRAM -> subsample context) ---
from tabpfn import TabPFNClassifier
from sklearn.model_selection import train_test_split

# holdout split of the META features — keep row indices so we can compare base vs meta fairly
all_idx = np.arange(len(y_np))
tr_idx, va_idx = train_test_split(all_idx, test_size=0.2, stratify=y_np, random_state=SEED)
mtr, mva   = meta_train[tr_idx], meta_train[va_idx]
ytr_m, yva_m = y_np[tr_idx], y_np[va_idx]

# TabPFN context must fit in 6 GB -> stratified subsample of the training part
CTX = 30000   # context rows fed to TabPFN (raise if VRAM allows, lower if OOM)
sub, _ = train_test_split(np.arange(len(ytr_m)), train_size=CTX,
                          stratify=ytr_m, random_state=SEED)
print(f"TabPFN context rows: {len(sub)}  (of {len(ytr_m)} train)")

t0 = time.time()
tabpfn = TabPFNClassifier(
    device='cuda',
    n_estimators=4,
    balance_probabilities=True,        # ref lesson: let this handle imbalance, no hard prior
    ignore_pretraining_limits=True,
)
tabpfn.fit(mtr[sub], ytr_m[sub])

# predict holdout in chunks (avoid VRAM spike)
def tabpfn_predict_proba(clf, Xq, chunk=20000):
    out = [clf.predict_proba(Xq[s:s+chunk]) for s in range(0, len(Xq), chunk)]
    return np.concatenate(out, axis=0)

proba_va = tabpfn_predict_proba(tabpfn, mva)
ba_meta = balanced_accuracy_score(yva_m, proba_va.argmax(1))
print(f"TabPFN meta holdout balanced_acc = {ba_meta:.4f}   ({time.time()-t0:.0f}s)")

# fair comparison on the SAME holdout rows
print("\n-- same-holdout references --")
for n in names:
    ba = balanced_accuracy_score(yva_m, oof_probs[n][va_idx].argmax(1))
    print(f"  base {n}: {ba:.4f}")
avg_va = sum(oof_probs[n][va_idx] for n in names) / len(names)
print(f"  avg-blend:   {balanced_accuracy_score(yva_m, avg_va.argmax(1)):.4f}")
print(f"  TabPFN meta: {ba_meta:.4f}")

In [ ]:
# --- Push TabPFN context size (more context = better, if VRAM holds) ---
# Reuses the same holdout (mtr/mva, ytr_m/yva_m) from the cell above.
for CTX_try in [60000, 100000, 160000]:
    try:
        sub2, _ = train_test_split(np.arange(len(ytr_m)), train_size=CTX_try,
                                   stratify=ytr_m, random_state=SEED)
        t0 = time.time()
        clf = TabPFNClassifier(device='cuda', n_estimators=4,
                               balance_probabilities=True, ignore_pretraining_limits=True)
        clf.fit(mtr[sub2], ytr_m[sub2])
        pv = tabpfn_predict_proba(clf, mva, chunk=20000)
        ba = balanced_accuracy_score(yva_m, pv.argmax(1))
        print(f"CTX={CTX_try:>6}: holdout bal_acc={ba:.4f}  ({time.time()-t0:.0f}s)")
        del clf; torch.cuda.empty_cache()
    except RuntimeError as e:
        print(f"CTX={CTX_try:>6}: FAILED ({str(e)[:60]}) -> VRAM limit, stop here")
        torch.cuda.empty_cache()
        break

## Final fit + submission

Context plateaus at ~0.964 (30k ≈ 160k), so we use **CTX=30000** (same accuracy, 15× faster).
Fit TabPFN on the full `meta_train` (30k stratified context), predict `meta_test`, write `submission.csv`.
Multi-seed context averaging reduces the variance from the random subsample.

In [ ]:
# --- Final: multi-seed TabPFN over full meta_train -> test probs -> submission ---
CTX_FINAL = 30000
SEEDS_FINAL = [42, 1, 7]   # average over a few context subsamples to cut variance

test_proba_sum = np.zeros((len(meta_test), N_CLASSES), dtype='float32')
t0 = time.time()
for s in SEEDS_FINAL:
    sub_f, _ = train_test_split(np.arange(len(y_np)), train_size=CTX_FINAL,
                                stratify=y_np, random_state=s)
    clf = TabPFNClassifier(device='cuda', n_estimators=4,
                           balance_probabilities=True, ignore_pretraining_limits=True)
    clf.fit(meta_train[sub_f], y_np[sub_f])
    test_proba_sum += tabpfn_predict_proba(clf, meta_test, chunk=20000) / len(SEEDS_FINAL)
    print(f"  seed {s} done  ({time.time()-t0:.0f}s)")
    del clf; torch.cuda.empty_cache()

final_pred_int = test_proba_sum.argmax(1)
final_pred_lbl = np.array([int_to_class[i] for i in final_pred_int])

# build submission
sub = pd.DataFrame({'id': df_test['id'].values, 'class': final_pred_lbl})
sub.to_csv('../submission.csv', index=False)
print("\nsubmission.csv written:", sub.shape)
print(sub['class'].value_counts())
print("\ntrain class proportions for reference:")
print(pd.Series(y_np).map(int_to_class).value_counts(normalize=True).round(4))
sub.head()

# ── v2: richer feature engineering (push toward 0.971) ──

First sub = **LB 0.96577**. Now widen the feature set the base models see:
- **All band-pair colors** (not just adjacent): u-r, u-i, u-z, g-i, g-z, r-z
- **redshift transforms**: log1p(redshift) compresses the QSO tail; redshift² for curvature
- **redshift × color interactions**: the STAR/GALAXY/QSO overlap is redshift-dependent
- **brightness aggregates**: mean magnitude, color spread (max−min)

Rebuild XGB / CatBoost / MLP OOF on the wider matrix, re-stack with TabPFN, resubmit.

In [ ]:
def build_features_v2(df):
    out = df.copy()
    bands = ['u', 'g', 'r', 'i', 'z']

    # all band-pair colors (10 pairs)
    for a in range(len(bands)):
        for b in range(a + 1, len(bands)):
            out[f'{bands[a]}_{bands[b]}'] = out[bands[a]] - out[bands[b]]

    # redshift transforms
    z = out['redshift']
    out['redshift_log1p'] = np.log1p(z.clip(lower=0))   # clip tiny negatives for log
    out['redshift_sq']    = z ** 2

    # redshift x color interactions (overlap zones depend on z)
    for col in ['u_g', 'g_r', 'r_i', 'i_z']:
        out[f'z_x_{col}'] = z * out[col]

    # brightness aggregates
    out['mag_mean'] = out[bands].mean(axis=1)
    out['mag_std']  = out[bands].std(axis=1)
    color_cols_all  = [f'{bands[a]}_{bands[b]}' for a in range(len(bands)) for b in range(a+1, len(bands))]
    out['color_spread'] = out[color_cols_all].max(axis=1) - out[color_cols_all].min(axis=1)

    # one-hot cats
    out = pd.get_dummies(out, columns=['spectral_type', 'galaxy_population'], dtype=float)
    return out

train_v2 = build_features_v2(df_train)
test_v2  = build_features_v2(df_test)

drop_cols = ['id', 'class']
feat_v2 = [c for c in train_v2.columns if c not in drop_cols]
X2      = train_v2[feat_v2].astype('float32')
X2_test = test_v2.reindex(columns=feat_v2, fill_value=0.0).astype('float32')

X2_np      = X2.values.astype('float32')
X2_test_np = X2_test.values.astype('float32')

print("v2 features:", len(feat_v2), "(was 18)")
print("X2:", X2_np.shape, "| X2_test:", X2_test_np.shape)
print("any NaN/inf?", np.isnan(X2_np).any(), np.isinf(X2_np).any(),
      np.isnan(X2_test_np).any(), np.isinf(X2_test_np).any())
print("\nnew cols:", [c for c in feat_v2 if c not in feature_cols])

In [ ]:
# --- v2 base models on the 33-feature matrix ---
oof_probs_v2  = {}
test_probs_v2 = {}

def run_oof_v2(name, make_model, fit_predict):
    oof  = np.zeros((len(y_np), N_CLASSES), dtype='float32')
    test = np.zeros((X2_test_np.shape[0], N_CLASSES), dtype='float32')
    t0 = time.time()
    for f, (tr, va) in enumerate(oof_skf.split(X2_np, y_np)):
        m = make_model()
        oof[va] = fit_predict(m, X2_np[tr], y_np[tr], X2_np[va])
        test   += fit_predict(m, X2_np[tr], y_np[tr], X2_test_np) / N_FOLDS
        print(f"  [{name}] fold {f+1}/{N_FOLDS} bal_acc="
              f"{balanced_accuracy_score(y_np[va], oof[va].argmax(1)):.4f} ({time.time()-t0:.0f}s)")
    overall = balanced_accuracy_score(y_np, oof.argmax(1))
    print(f"  [{name}] OOF balanced_acc = {overall:.4f}  total {time.time()-t0:.0f}s")
    oof_probs_v2[name]  = oof
    test_probs_v2[name] = test

print("=== XGBoost v2 ===")
run_oof_v2('xgb', make_xgb, xgb_fit_predict)
print("=== CatBoost v2 ===")
run_oof_v2('cat', make_cat, cat_fit_predict)
print("=== MLP v2 ===")
run_oof_v2('mlp', make_mlp, mlp_fit_predict)

In [ ]:
# --- v2 meta features + TabPFN holdout check ---
names_v2 = ['xgb','cat','mlp']
meta_train_v2 = np.concatenate(
    [to_logit(oof_probs_v2[n]) for n in names_v2] + [X2_np], axis=1).astype('float32')
meta_test_v2 = np.concatenate(
    [to_logit(test_probs_v2[n]) for n in names_v2] + [X2_test_np], axis=1).astype('float32')
print("meta_train_v2:", meta_train_v2.shape, "| meta_test_v2:", meta_test_v2.shape)

# holdout (same split indices as v1 for fair compare)
tr_idx2, va_idx2 = train_test_split(np.arange(len(y_np)), test_size=0.2,
                                    stratify=y_np, random_state=SEED)
mtr2, mva2   = meta_train_v2[tr_idx2], meta_train_v2[va_idx2]
ytr2, yva2   = y_np[tr_idx2], y_np[va_idx2]

CTX = 30000
sub2, _ = train_test_split(np.arange(len(ytr2)), train_size=CTX, stratify=ytr2, random_state=SEED)
t0=time.time()
clf = TabPFNClassifier(device='cuda', n_estimators=4, balance_probabilities=True,
                       ignore_pretraining_limits=True)
clf.fit(mtr2[sub2], ytr2[sub2])
pv2 = tabpfn_predict_proba(clf, mva2, chunk=20000)
ba_v2 = balanced_accuracy_score(yva2, pv2.argmax(1))
print(f"TabPFN meta v2 holdout balanced_acc = {ba_v2:.4f}  ({time.time()-t0:.0f}s)")
print(f"  (v1 meta holdout was 0.9642 -> LB 0.96577)")
del clf; torch.cuda.empty_cache()

In [ ]:
# --- v2 final submission: multi-seed TabPFN on full meta_train_v2 ---
CTX_FINAL = 30000
SEEDS_FINAL = [42, 1, 7]
test_proba_v2 = np.zeros((len(meta_test_v2), N_CLASSES), dtype='float32')
t0=time.time()
for s in SEEDS_FINAL:
    sub_f, _ = train_test_split(np.arange(len(y_np)), train_size=CTX_FINAL,
                                stratify=y_np, random_state=s)
    clf = TabPFNClassifier(device='cuda', n_estimators=4, balance_probabilities=True,
                           ignore_pretraining_limits=True)
    clf.fit(meta_train_v2[sub_f], y_np[sub_f])
    test_proba_v2 += tabpfn_predict_proba(clf, meta_test_v2, chunk=20000) / len(SEEDS_FINAL)
    print(f"  seed {s} done ({time.time()-t0:.0f}s)")
    del clf; torch.cuda.empty_cache()

pred_v2 = np.array([int_to_class[i] for i in test_proba_v2.argmax(1)])
sub_v2 = pd.DataFrame({'id': df_test['id'].values, 'class': pred_v2})
sub_v2.to_csv('../submission_v2.csv', index=False)
print("\nsubmission_v2.csv written:", sub_v2.shape)
print(sub_v2['class'].value_counts())
# how many predictions changed vs v1 submission?
import os
if os.path.exists('../submission.csv'):
    v1 = pd.read_csv('../submission.csv')
    changed = (v1['class'].values != pred_v2).sum()
    print(f"\nflips vs v1 submission: {changed} ({100*changed/len(pred_v2):.2f}%)")
sub_v2.head()

In [ ]:
# --- v3: add LightGBM as 4th base (GPU), on the 33-feature matrix ---
import lightgbm as lgb

def make_lgb():
    return lgb.LGBMClassifier(
        n_estimators=1200,
        learning_rate=0.05,
        num_leaves=63,
        max_depth=-1,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        min_child_samples=40,
        objective='multiclass',
        num_class=N_CLASSES,
        device='gpu',
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )

def lgb_fit_predict(model, Xtr, ytr, Xpred):
    model.fit(Xtr, ytr)
    return model.predict_proba(Xpred)

print("=== LightGBM v3 (GPU) ===")
run_oof_v2('lgb', make_lgb, lgb_fit_predict)   # stores into oof_probs_v2 / test_probs_v2

In [ ]:
# --- v3 meta: 4 bases (xgb,cat,mlp,lgb) + 33 raw feats -> TabPFN holdout ---
names_v3 = ['xgb','cat','mlp','lgb']
meta_train_v3 = np.concatenate(
    [to_logit(oof_probs_v2[n]) for n in names_v3] + [X2_np], axis=1).astype('float32')
meta_test_v3 = np.concatenate(
    [to_logit(test_probs_v2[n]) for n in names_v3] + [X2_test_np], axis=1).astype('float32')
print("meta_train_v3:", meta_train_v3.shape, "| meta_test_v3:", meta_test_v3.shape)

tr_idx3, va_idx3 = train_test_split(np.arange(len(y_np)), test_size=0.2,
                                    stratify=y_np, random_state=SEED)
mtr3, mva3 = meta_train_v3[tr_idx3], meta_train_v3[va_idx3]
ytr3, yva3 = y_np[tr_idx3], y_np[va_idx3]

CTX = 30000
sub3, _ = train_test_split(np.arange(len(ytr3)), train_size=CTX, stratify=ytr3, random_state=SEED)
t0=time.time()
clf = TabPFNClassifier(device='cuda', n_estimators=4, balance_probabilities=True,
                       ignore_pretraining_limits=True)
clf.fit(mtr3[sub3], ytr3[sub3])
pv3 = tabpfn_predict_proba(clf, mva3, chunk=20000)
ba_v3 = balanced_accuracy_score(yva3, pv3.argmax(1))
print(f"TabPFN meta v3 holdout balanced_acc = {ba_v3:.4f}  ({time.time()-t0:.0f}s)")
print(f"  (v1=0.9642 -> LB 0.96577 | v2=0.9646 -> LB 0.96662)")
del clf; torch.cuda.empty_cache()

In [ ]:
# --- v3 final submission: multi-seed TabPFN on full meta_train_v3 (4 bases) ---
CTX_FINAL = 30000
SEEDS_FINAL = [42, 1, 7]
test_proba_v3 = np.zeros((len(meta_test_v3), N_CLASSES), dtype='float32')
t0=time.time()
for s in SEEDS_FINAL:
    sub_f, _ = train_test_split(np.arange(len(y_np)), train_size=CTX_FINAL,
                                stratify=y_np, random_state=s)
    clf = TabPFNClassifier(device='cuda', n_estimators=4, balance_probabilities=True,
                           ignore_pretraining_limits=True)
    clf.fit(meta_train_v3[sub_f], y_np[sub_f])
    test_proba_v3 += tabpfn_predict_proba(clf, meta_test_v3, chunk=20000) / len(SEEDS_FINAL)
    print(f"  seed {s} done ({time.time()-t0:.0f}s)")
    del clf; torch.cuda.empty_cache()

pred_v3 = np.array([int_to_class[i] for i in test_proba_v3.argmax(1)])
sub_v3 = pd.DataFrame({'id': df_test['id'].values, 'class': pred_v3})
sub_v3.to_csv('../submission_v3.csv', index=False)
print("\nsubmission_v3.csv written:", sub_v3.shape)
print(sub_v3['class'].value_counts())
import os
if os.path.exists('../submission_v2.csv'):
    v2 = pd.read_csv('../submission_v2.csv')
    flips = (v2['class'].values != pred_v3).sum()
    print(f"\nflips vs v2: {flips} ({100*flips/len(pred_v3):.2f}%)")
sub_v3.head()

In [ ]:
# --- v4 tuning: TabPFN n_estimators + context sweep on v3 holdout ---
# reuses mtr3/mva3, ytr3/yva3 from the v3 meta cell
for NE, CTX in [(8, 30000), (16, 30000), (8, 50000), (16, 50000)]:
    try:
        subN, _ = train_test_split(np.arange(len(ytr3)), train_size=CTX,
                                   stratify=ytr3, random_state=SEED)
        t0=time.time()
        clf = TabPFNClassifier(device='cuda', n_estimators=NE, balance_probabilities=True,
                               ignore_pretraining_limits=True)
        clf.fit(mtr3[subN], ytr3[subN])
        pv = tabpfn_predict_proba(clf, mva3, chunk=20000)
        ba = balanced_accuracy_score(yva3, pv.argmax(1))
        print(f"n_est={NE:>2} CTX={CTX:>6}: holdout bal_acc={ba:.4f}  ({time.time()-t0:.0f}s)")
        del clf; torch.cuda.empty_cache()
    except RuntimeError as e:
        print(f"n_est={NE} CTX={CTX}: OOM/FAIL ({str(e)[:50]})")
        torch.cuda.empty_cache()
print("(v3 baseline was n_est=4 CTX=30k -> 0.9653)")

In [ ]:
# --- v4: stronger + multi-seed base models on 33-feature matrix ---
BASE_SEEDS = [42, 1, 7]
oof_v4  = {}
test_v4 = {}

def run_oof_multiseed(name, make_model_seeded, fit_predict, seeds=BASE_SEEDS):
    """Average OOF/test over multiple seeds -> lower-variance base signal."""
    oof  = np.zeros((len(y_np), N_CLASSES), dtype='float32')
    test = np.zeros((X2_test_np.shape[0], N_CLASSES), dtype='float32')
    t0 = time.time()
    for si, sd in enumerate(seeds):
        for f, (tr, va) in enumerate(oof_skf.split(X2_np, y_np)):
            m = make_model_seeded(sd)
            oof[va] += fit_predict(m, X2_np[tr], y_np[tr], X2_np[va]) / len(seeds)
            test    += fit_predict(m, X2_np[tr], y_np[tr], X2_test_np) / (len(seeds)*N_FOLDS)
        print(f"  [{name}] seed {sd} done ({time.time()-t0:.0f}s)")
    ba = balanced_accuracy_score(y_np, oof.argmax(1))
    print(f"  [{name}] multi-seed OOF balanced_acc = {ba:.4f}  total {time.time()-t0:.0f}s")
    oof_v4[name]  = oof
    test_v4[name] = test

# stronger XGB
def make_xgb4(sd):
    return xgb.XGBClassifier(n_estimators=1500, learning_rate=0.03, max_depth=8,
        subsample=0.85, colsample_bytree=0.8, min_child_weight=3, reg_lambda=1.5,
        objective='multi:softprob', num_class=N_CLASSES, tree_method='hist',
        device='cuda', eval_metric='mlogloss', random_state=sd)
# stronger LGB
def make_lgb4(sd):
    return lgb.LGBMClassifier(n_estimators=1800, learning_rate=0.03, num_leaves=127,
        subsample=0.85, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.5,
        min_child_samples=40, objective='multiclass', num_class=N_CLASSES,
        device='gpu', random_state=sd, n_jobs=-1, verbose=-1)
# stronger Cat
def make_cat4(sd):
    return CatBoostClassifier(iterations=2000, learning_rate=0.03, depth=8,
        l2_leaf_reg=3.0, loss_function='MultiClass', task_type='GPU', devices='0',
        random_seed=sd, verbose=False, allow_writing_files=False)

print("=== XGB v4 (stronger, 3 seeds) ===");  run_oof_multiseed('xgb', make_xgb4, xgb_fit_predict)
print("=== LGB v4 (stronger, 3 seeds) ===");  run_oof_multiseed('lgb', make_lgb4, lgb_fit_predict)
print("=== Cat v4 (stronger, 3 seeds) ===");  run_oof_multiseed('cat', make_cat4, cat_fit_predict)
# MLP: reuse v2 single-seed result (already in oof_probs_v2) to save time
oof_v4['mlp']  = oof_probs_v2['mlp']
test_v4['mlp'] = test_probs_v2['mlp']
print("MLP: reused v2 OOF (0.9506)")